# Extraction des Caractéristiques (Embeddings) via Deep Learning

Ce notebook constitue l'étape de **Feature Extraction**. Puisque nous travaillons avec des images complexes (IRM), nous ne pouvons pas utiliser directement les pixels bruts pour du clustering ou de la classification simple.

**Le concept :**
Nous utilisons un réseau de neurones pré-entraîné (**ResNet-50**) comme un "extracteur de caractéristiques". Nous supprimons la couche finale de classification pour ne récupérer que le "Résumé Numérique" de l'image (l'embedding). 

L'embedding est un vecteur de **2048 nombres** qui représente l'image dans un espace mathématique de haute dimension, capturant les formes, textures et structures anatomiques.

---
# 1. Préparation de l'environnement

In [1]:
# Imports des bibliothèques nécessaires
import sys
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm # Pour les barres de progression
import torch # Cœur de la bibliothèque de Deep Learning
import torch.nn as nn # Pour manipuler les couches des réseaux
from torchvision import models, transforms # Modèles pré-entraînés et outils de transformation
import numpy as np

# Ajout du dossier racine au chemin système pour importer nos propres modules (config)
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from config.config import *

# 2. Pipeline de Preprocessing
Pour qu'un réseau de neurones comprenne nos images, elles doivent toutes avoir le même format et suivre les mêmes statistiques que celles utilisées lors de l'entraînement du modèle original (ImageNet).

In [3]:
# Définition de la suite de transformations (Pipeline)
transform = transforms.Compose([
    # Redimensionne l'image en 224x224 pixels (format standard pour ResNet)
    transforms.Resize((224, 224)),
    
    # Transforme l'image PIL en "Tensor" (format mathématique pour PyTorch)
    # Et convertit les pixels (0-255) en valeurs entre 0 et 1
    transforms.ToTensor(),
    
    # Normalise les couleurs avec la moyenne (mean) et l'écart-type (std)
    # Ces valeurs sont spécifiques aux modèles entraînés sur le dataset ImageNet
    # Cela aide le modèle à converger plus rapidement
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 3. Chargement et Adaptation du Modèle
Nous allons charger un modèle **ResNet-50** déjà "intelligent" (ayant appris sur des millions d'images) et le modifier pour qu'il nous donne des vecteurs au lieu de labels de classes.

In [4]:
# 1. Chargement du modèle avec ses poids pré-entraînés
# ResNet50 est un réseau très profond (50 couches) excellent pour la reconnaissance de formes
resnet = models.resnet50(pretrained=True)

# 2. Passage en mode évaluation
# Cela désactive certaines couches comme le 'Dropout' ou la 'BatchNormalization' qui ne servent qu'à l'entraînement
resnet.eval()

# 3. Gel des paramètres (Freeze)
# On dit à PyTorch de ne pas calculer de gradients : on ne veut pas ré-entraîner le modèle, juste l'utiliser.
for param in resnet.parameters():
    param.requires_grad = False

# 4. Création de l'extracteur de caractéristiques
# resnet.children() liste toutes les couches du modèle. 
# [:-1] signifie : "Prends tout sauf la dernière couche" (celle qui classe en 1000 catégories).
# nn.Sequential réassemble ces couches dans un nouveau modèle "tronqué".
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])

c:\Users\Fabien\Desktop\OC\P10\BrainScanAI\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Fabien\Desktop\OC\P10\BrainScanAI\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
# Fonction utilitaire pour transformer une image en vecteur (embedding)
def extract_features(image_path, model, transform) -> list :
    # Charge l'image et s'assure qu'elle est en mode RGB (3 canaux)
    img = Image.open(image_path).convert('RGB')
    
    # Applique le preprocessing et ajoute une dimension "Batch" (1, 3, 224, 224)
    # PyTorch attend toujours un lot d'images, même s'il n'y en a qu'une.
    image = transform(img).unsqueeze(0)
    
    # Désactive le calcul de gradient pour économiser la mémoire et accélérer le calcul
    with torch.no_grad():
        features = model(image)

    # .squeeze() retire les dimensions inutiles (le 1 du batch)
    # .numpy() convertit le tensor PyTorch en tableau NumPy
    # .tolist() transforme le tout en liste standard pour le stockage en CSV/Parquet
    return features.squeeze().numpy().tolist()

In [6]:
images_cancer_dir = BASE_DIR / "mri_dataset_brain_cancer_oc" / "avec_labels" / "cancer"
images_normal_dir = BASE_DIR / "mri_dataset_brain_cancer_oc" / "avec_labels" / "normal"
images_sans_label_dir = BASE_DIR / "mri_dataset_brain_cancer_oc" / "sans_label"

# 4. Exécution de l'Extraction
Nous allons maintenant appliquer ce processus à toutes les images de nos dossiers (Cancer, Normal et Sans Label) et sauvegarder le résultat.

In [7]:
# Définition du chemin de sauvegarde (Format Parquet : plus léger et rapide que le CSV)
features_path = BASE_DIR / "mri_dataset_brain_cancer_oc" / "features.parquet"

# Vérification si le travail a déjà été fait auparavant
if features_path.exists():
    print(f"--- Fichier trouvé : {features_path} ---")
    print("Chargement des embeddings existants...")
    df_features = pd.read_parquet(features_path)
    features_already_present = True
else:
    print("--- Fichier non trouvé : Lancement du traitement lourd ---")
    features = []

    # Liste des dossiers à traiter
    dossiers_a_scanner = [images_cancer_dir, images_normal_dir, images_sans_label_dir]

    # Boucle sur chaque dossier avec barre de progression principale
    for path in tqdm(dossiers_a_scanner, desc="Dossiers en cours"):
        # Récupération de la liste des fichiers .jpg
        liste_images = list(path.glob('*.jpg'))
        
        # Boucle sur chaque image du dossier
        for image_path in tqdm(liste_images, desc=f"Extraction {path.name}", leave=False):
            try:
                # Création d'un dictionnaire avec le nom du fichier et son vecteur mathématique
                data = {
                    'name' : image_path.name,
                    'features' : extract_features(image_path, feature_extractor, transform)
                }
                features.append(data)
            except Exception as e:
                # En cas d'image corrompue ou d'erreur, on affiche un avertissement sans arrêter le script
                print(f"❌ Erreur sur {image_path.name}: {e}")

    # Création du tableau final (DataFrame)
    df_features = pd.DataFrame(features)

    if not df_features.empty:
        # Vérification du nombre de dimensions (doit être 2048 pour ResNet50)
        dim_size = len(df_features['features'].iloc[0])
        print(f"✅ Extraction terminée. Chaque image est représentée par {dim_size} caractéristiques.")

    # Sauvegarde au format Parquet pour une utilisation future hyper-efficace
    df_features.to_parquet(features_path)
    print(f"📁 Embeddings enregistrés avec succès dans : {features_path}")
    features_already_present = False

# Petit aperçu des données générées
df_features.head()

--- Fichier trouvé : C:\Users\Fabien\Desktop\OC\P10\BrainScanAI\mri_dataset_brain_cancer_oc\features.parquet ---
Chargement des embeddings existants...


,name,features
0,05340cd4-3bb2-459d-9937-bf27d52d8351.jpg,"[0.40974900126457214, 0.8732983469963074, 0.17..."
1,0c6f3641-60d9-4a76-abe5-de89d55d5f2c.jpg,"[0.6502299308776855, 0.721211314201355, 0.0541..."
2,0f718241-8f63-4b55-81ce-315324b51069.jpg,"[1.192965030670166, 1.1314507722854614, 0.1077..."
3,11a7a426-4806-401e-98b2-b96e7094d1a6.jpg,"[0.4946941137313843, 0.4560645818710327, 0.174..."
4,1c043dbb-4623-4769-8e5e-0223bd745040.jpg,"[0.4919051229953766, 0.35454875230789185, 0.39..."
